# Dual-Stream Deepfake Detector — Colab Training Driver
Run cells top to bottom. Runtime > Change runtime type > GPU (T4) first.

In [ ]:
!git clone https://github.com/<your-username>/dual-stream-deepfake-detection.git
%cd dual-stream-deepfake-detection
!pip install -e .[colab]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/data'  # or a path under /content/drive/MyDrive for persistence

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json here
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!python scripts/download_datasets.py \
  --kaggle-dataset xhlulu/140k-real-and-fake-faces \
  --out-dir $DATA_DIR

## StyleGAN3 synthetic augmentation
Clone NVlabs/stylegan3 and download a pretrained checkpoint (see README) before running this cell.

In [ ]:
!git clone https://github.com/NVlabs/stylegan3.git
!python scripts/generate_stylegan3.py \
  --stylegan3-repo stylegan3 \
  --checkpoint /content/drive/MyDrive/checkpoints/stylegan3-r-ffhq-1024x1024.pkl \
  --out-dir $DATA_DIR/stylegan3 \
  --num-images 5000

## Build manifests, then train baseline and dual-stream models
Build `original_manifest.csv`, `augmented_manifest.csv`, and `test_manifest.csv` from the downloaded folders using `dsfnet.data.manifest` (see README 'Building manifests'), then train:

In [ ]:
!python scripts/train_model.py --manifest $DATA_DIR/original_manifest.csv --model spatial_only --out-dir runs/baseline
!python scripts/train_model.py --manifest $DATA_DIR/augmented_manifest.csv --model dual_stream --out-dir runs/dual_stream

In [ ]:
!python scripts/evaluate_models.py \
  --test-manifest $DATA_DIR/test_manifest.csv \
  --baseline-checkpoint runs/baseline/spatial_only.pth --baseline-model spatial_only \
  --candidate-checkpoint runs/dual_stream/dual_stream.pth --candidate-model dual_stream \
  --out-report runs/fpr_comparison_report.json